# Experimento — TabKAN

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

KAN desenhada para tabela, no sentido de Eslamian, Afzal Aghaei e Cheng (2025). O modelo troca a base funcional da aresta conforme a variante: spline, Chebyshev, Jacobi, Padé, Fourier, fracionária ou FastKAN. No código, o mixer existe para spline, Chebyshev e Fourier. A classificação de `Churn` usa uma dessas variantes, treinada do zero nesta base.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

O artigo busca, por família, profundidade, largura e o parâmetro da base. O Blastchar desse artigo tem 7.043 linhas: é esta tabela de churn. A escolha deles é o melhor F1 de validação. Taxa de aprendizagem, troca de otimizador, épocas, batch, dropout e a troca entre logística, tangente hiperbólica e ReLU não entram nessa busca. O protocolo fixa L-BFGS, lote completo e entropia cruzada.

As famílias da comparação final entram uma por bloco: spline, Chebyshev, Jacobi, Padé, Fourier, fracionária e FastKAN. wav-KAN e fc-KAN saíram da tabela final e ficam de fora. Mixer, no código, existe para spline, Chebyshev e Fourier, e só roda depois que essa família já tiver vencido sem mixer.

A faixa publicada na seção 3 vale para a família que vencer:

- Profundidade. Chebyshev, Jacobi, fracionária e spline: 1 a 10. Padé, Fourier e FastKAN: 1 a 5.
- Largura, de 5 em 5. Chebyshev, Jacobi, fracionária, Padé e spline: 5 a 100. Fourier e FastKAN: 5 a 50.
- Spline: `grid` de 1 a 9, de 2 em 2, e `k` de 1 a 5. São dois blocos.
- Chebyshev, Jacobi e fracionária: ordem do polinômio de 2 a 6. Na fracionária, o ν é peso treinável, não é argumento da busca.
- Padé: os graus q e k, os dois de 2 a 6, em blocos separados.
- Fourier: `gridsize` de 1 a 10.
- FastKAN: a chamada no repositório só recebe `width`. Não há parâmetro próprio para mais um bloco.

Logística, tangente hiperbólica e ReLU não são argumentos de nenhuma família. No Chebyshev, `tanh` está fixo dentro da camada para levar a entrada ao intervalo [−1, 1]. No spline, a base residual do código usado é SiLU, e a busca não a varia.

A transferência da seção 4.4 não pede outra tabela. Parte a mesma tabela em dois conjuntos com metade das colunas em comum, congela tudo menos a camada final e depois inverte os papéis. Isso é um bloco de dados, e só abre se for proposto. Não é hiperparâmetro do modelo.

O oversampling repetitivo das dobras fica ligado no bloco 0, por causa do pré-processamento comum. O artigo não o usa; o bloco anota isso.

Fontes: arXiv:2504.06559 e os scripts `runs/NAS-*.py` em github.com/aseslamian/TAbKAN.


## Hiperparâmetros que ainda vão chegar

A pesquisa destes hiperparâmetros está nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. Spline, rasa, L-BFGS, lote completo, entropia cruzada, `grid` e `k` no default da classe (`grid` 5, `k` 3), sobre as dobras com oversampling.
2. As outras famílias, uma por bloco: Chebyshev, Jacobi, Padé, Fourier, fracionária e FastKAN.
3. Profundidade da família vencedora, na faixa da seção 3.
4. Largura, de 5 em 5, na faixa dessa família.
5. Parâmetro da base, se a família tiver: spline primeiro `grid` e depois `k`; Chebyshev, Jacobi ou fracionária a ordem; Padé primeiro q e depois k; Fourier o `gridsize`. FastKAN para aqui.
6. Mixer, só se a vencedora for spline, Chebyshev ou Fourier.
7. Transferência da seção 4.4, só se for proposta.
8. Estratégias de dados compartilhadas.
9. Bloco final no teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
